# Day 2: RDD Practice Solutions

## Welcome!
These are the solutions for the exercises in `02_exercise.ipynb`. Use them to check your work.

## Before You Start
- Ensure you have run `docker-compose up` in the `01_basic_spark` directory.
- Open Jupyter Notebook at `http://localhost:8888`.
- Place the `covid-19.csv` file in your data directory (e.g., `covid-dataset/covid-data.csv`).
- Download the CSV from: https://raw.githubusercontent.com/owid/covid-19-data/master/public/data/owid-covid-data.csv
- Expected output may vary based on dataset updates.

## Solutions

### Exercise 1: Find the Average of Numbers
Make an RDD from `[5, 10, 15, 20, 25]` and compute the average value.


In [ ]:
from pyspark import SparkContext

# Initialize Spark context
sc = SparkContext.getOrCreate()

# Create an RDD from a Python list
numbers = [5, 10, 15, 20, 25]
rdd = sc.parallelize(numbers)  # parallelize converts list to RDD

# Compute the sum of all numbers using reduce
total_sum = rdd.reduce(lambda x, y: x + y)  # reduce applies the lambda cumulatively

# Count the total number of elements in the RDD
count = rdd.count() 

# Compute the average
average = total_sum / count

# Print the result
print("Average:", average)  # Output: Average: 15.0


**Expected Output**: `15.0`

### Exercise 2: Count Distinct Numbers
Create an RDD from `[1, 2, 2, 3, 4, 4, 5]` and count how many distinct numbers are present.


In [ ]:
from pyspark import SparkContext

# Initialize Spark context
sc = SparkContext.getOrCreate()

# Create an RDD from a Python list
numbers = [1, 2, 2, 3, 4, 4, 5]
rdd = sc.parallelize(numbers)  # Convert list to RDD

# Get distinct values and count them
distinct_count = rdd.distinct().count()  
# distinct() removes duplicates
# count() returns total unique elements

# Print the result
print("Number of distinct values:", distinct_count)


**Expected Output**: `5`

### Exercise 3: Find Maximum from RDD
Extract `[3, 8, 2, 10, 6]` and find the maximum number.

In [ ]:
from pyspark import SparkContext

# Initialize Spark context
sc = SparkContext.getOrCreate()

# Create an RDD from a Python list
numbers = [3, 8, 2, 10, 6]
rdd = sc.parallelize(numbers)  # Convert list to RDD

# Find the maximum value using reduce
max_num = rdd.reduce(lambda a, b: a if a > b else b)  
# Compare elements pairwise, keep the larger one

# Print the maximum value
print("Maximum number:", max_num)


**Expected Output** : `10`

### Exercise 4: Read CSV with RDD and Compute KPI
Load `covid-19.csv`, filter for Afghanistan (iso_code = 'AFG'), and compute total `new_cases`.

In [ ]:
from pyspark import SparkContext

# Initialize Spark context
sc = SparkContext.getOrCreate()

# Load CSV file into an RDD (each line is a string)
rdd = sc.textFile("covid-dataset/covid-data.csv")  

# Get the header row
header = rdd.first()  

# Filter out header and keep only Afghanistan data (iso_code == 'AFG')
afg_data = rdd.filter(lambda row: row != header and row.startswith("AFG"))  

# Extract 'new_cases' column (6th column) and convert to float
new_cases = afg_data.map(lambda row: float(row.split(',')[5] or 0.0))  

# Compute total new cases
total_cases = new_cases.sum()  

# Display total new cases
print(f"Total new cases for Afghanistan: {total_cases}")


**Expected Output**: `Total new cases for Afghanistan: ~235214.0` (depends on dataset)

### Exercise 5: Factorials of Numbers
Create an RDD `[1, 2, 3, 4, 5]`, compute factorial of each number, then compute the total sum of factorials.

In [ ]:
from pyspark import SparkContext
import math

# Initialize Spark context
sc = SparkContext.getOrCreate()

# Create an RDD from a Python list
numbers = [1, 2, 3, 4, 5]
rdd = sc.parallelize(numbers)  # parallelize converts list to RDD

# Compute factorial of each number and sum them
factorial_sum = (
    rdd
    .map(lambda x: math.factorial(x))  # map: compute factorial of each element
    .reduce(lambda a, b: a + b)       # reduce: sum all factorials
)

# Print the result
print("Sum of factorials:", factorial_sum)


**Expected Output**: `(1! + 2! + 3! + 4! + 5! = 153)`

### Exercise 6: Sum of Cubes of Odd Numbers
Create an RDD `[1, 2, 3, 4, 5, 6, 7]`, filter odd numbers, cube them, and compute the sum.

In [ ]:
from pyspark import SparkContext

sc = SparkContext.getOrCreate()

numbers = [1, 2, 3, 4, 5, 6, 7]
rdd = sc.parallelize(numbers)

# Keep only odd numbers, cube them, and sum the results
result = (
    rdd
    .filter(lambda x: x % 2 != 0)   # Odd numbers only
    .map(lambda x: x**3)            # Cube each
    .reduce(lambda a, b: a + b)     # Sum all cubes
)

print("Sum of cubes of odd numbers:", result) 

**Expected Output**: Sum of cubes of odd numbers: `496`

### Exercise 7: Group by Continent
Load `covid-19.csv`, group by `continent`, and count records per continent.

In [ ]:
from pyspark import SparkContext

sc = SparkContext.getOrCreate()

rdd = sc.textFile("covid-dataset/covid-data.csv")  # Load CSV
header = rdd.first()  # Get header
counts = (rdd
          .filter(lambda row: row != header)               # Skip header
          .map(lambda row: (row.split(',')[1], 1))        # (continent, 1)
          .reduceByKey(lambda a, b: a + b))               # Sum per continent

print(counts.collect())


**Expected Output**: `[('Asia', X), ('Europe', Y), ...]` (counts depend on dataset)

### Exercise 8: Join Two RDDs
Join RDDs: `[('S1', 'Alice'), ('S2', 'Bob'), ('S3', 'Charlie')]` and `[('S1', 85), ('S2', 90), ('S4', 95)]`.

In [ ]:
from pyspark import SparkContext

sc = SparkContext.getOrCreate()

students = sc.parallelize([('S1','Alice'), ('S2','Bob'), ('S3','Charlie')])  # RDD of student IDs & names
grades = sc.parallelize([('S1',85), ('S2',90), ('S4',95)])                 # RDD of student IDs & grades

joined_rdd = students.join(grades)  # Join on student ID (key)

print(joined_rdd.collect())  # Collect joined RDD


**Expected Output**: `[('S1', ('Alice', 85)), ('S2', ('Bob', 90))]`

### Exercise 9: Cache RDD and Compute Average
Load `covid-19.csv`, filter for Brazil (iso_code = 'BRA'), cache, and compute average `new_deaths`.

In [ ]:
from pyspark import SparkContext

sc = SparkContext.getOrCreate()

rdd = sc.textFile("covid-dataset/covid-data.csv")  # Load CSV
header = rdd.first()                               # Get header
bra_data = rdd.filter(lambda row: row != header and row.startswith("BRA")).cache()  # Filter Brazil, cache

new_deaths = bra_data.map(lambda row: float(row.split(',')[6] or 0.0))  # Extract new_deaths column
count = new_deaths.count()                                             # Count rows
total = new_deaths.sum()                                               # Sum of new_deaths
average = total / count if count > 0 else 0.0                          # Compute average

print(f"Average new deaths for Brazil: {average}")


**Expected Output**: `Average new deaths for Brazil: 22408.555053763437`

### Exercise 10: Transform and Sort
Create an RDD from `[('apple', 5), ('banana', 2), ('orange', 8), ('apple', 3)]`, sum by key, and sort by value descending.

In [ ]:
from pyspark import SparkContext

sc = SparkContext.getOrCreate()

data = [('apple',5), ('banana',2), ('orange',8), ('apple',3)]
rdd = sc.parallelize(data)                        # Create RDD of key-value pairs

summed_rdd = rdd.reduceByKey(lambda a, b: a + b)  # Sum values by key
sorted_rdd = summed_rdd.sortBy(lambda x: x[1], ascending=False)  # Sort by value descending

print(sorted_rdd.collect())  # Collect sorted RDD


**Expected Output**: `[('orange', 8), ('apple', 8), ('banana', 2)]`

### Exercise 11: Filter by Date Range
Load `covid-19.csv`, filter for 2020, and count records.

In [ ]:
from pyspark import SparkContext

sc = SparkContext.getOrCreate()

rdd = sc.textFile("covid-dataset/covid-data.csv")  # Load CSV
header = rdd.first()                               # Get header
data_2020 = rdd.filter(lambda row: row != header and row.split(',')[3].startswith('2020'))  # Filter 2020

count = data_2020.count()  # Count rows

print(f"Number of records in 2020: {count}")


**Expected Output**: `Number of records in 2020: [90982]`

### Exercise 12: Top 5 Countries by Total Cases
Load `covid-19.csv`, find max `total_cases` per country, and get top 5.

In [ ]:
from pyspark import SparkContext

sc = SparkContext.getOrCreate()

rdd = sc.textFile("covid-dataset/covid-data.csv")  # Load CSV
header = rdd.first()                               # Get header
cases = rdd.filter(lambda row: row != header)     \
           .map(lambda row: (row.split(',')[2], float(row.split(',')[4] or 0.0)))  # (country, total_cases)

max_cases = cases.reduceByKey(lambda a, b: a if a > b else b)  # Max cases per country
top_5 = max_cases.top(5, key=lambda x: x[1])                   # Top 5 countries by cases

print(top_5)


**Expected Output**: `[('World', 775866783.0), ('High-income countries', 429044049.0), ('Asia', 301499099.0), ('Europe', 252916868.0), ('Upper-middle-income countries', 251753518.0)]` (values depend on dataset)

### Exercise 13: Combine Two Text RDDs
Combine RDDs from "spark is awesome" and "spark is fast", count unique words.

In [ ]:
from pyspark import SparkContext

sc = SparkContext.getOrCreate()

text1 = sc.parallelize("spark is awesome".split())  # RDD from first text
text2 = sc.parallelize("spark is fast".split())     # RDD from second text

combined_rdd = text1.union(text2)                   # Combine both RDDs
word_counts = combined_rdd.map(lambda w: (w, 1))   \
                          .reduceByKey(lambda a, b: a + b)  # Count occurrences

print(word_counts.collect())


**Expected Output**: `[('spark', 2), ('is', 2), ('awesome', 1), ('fast', 1)]`

### Exercise 14: Missing Values Count
Load `covid-19.csv`, count rows where `new_cases` is empty or non-numeric.

In [ ]:
from pyspark import SparkContext

sc = SparkContext.getOrCreate()

rdd = sc.textFile("covid-dataset/covid-data.csv")  # Load CSV
header = rdd.first()                               # Get header

# Filter rows with empty or non-numeric 'new_cases' (6th column)
invalid_cases = rdd.filter(lambda row: row != header) \
                   .filter(lambda row: not row.split(',')[5] or not row.split(',')[5].replace('.', '').replace('-', '').isdigit())

count = invalid_cases.count()  # Count invalid rows

print(f"Number of rows with missing or non-numeric new_cases: {count}")


**Expected Output**: `Number of rows with missing or non-numeric new_cases: 19276`

### Exercise 15: FlatMap and Distinct Words
Create an RDD from `["Spark is fun", "RDDs are powerful", "Spark uses RDDs"]`, split into words, and get distinct words.

In [ ]:
from pyspark import SparkContext

sc = SparkContext.getOrCreate()

sentences = ["Spark is fun", "RDDs are powerful", "Spark uses RDDs"]
rdd = sc.parallelize(sentences)              # Create RDD from list of sentences
words = rdd.flatMap(lambda s: s.split()).distinct()    # Split sentences into words and get unique words

print(words.collect())


**Expected Output**: `['Spark', 'is', 'fun', 'RDDs', 'are', 'powerful', 'uses']` (order may vary)

### Exercise 16: Aggregate Function for Sum and Count
Create an RDD from `[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]`, compute sum and count.

In [ ]:
from pyspark import SparkContext

sc = SparkContext.getOrCreate()

numbers = [1,2,3,4,5,6,7,8,9,10]
rdd = sc.parallelize(numbers)  

# Aggregate to compute sum and count in one pass
result = rdd.aggregate(
    (0,0),                                           # Initial value (sum, count)
    lambda acc, v: (acc[0]+v, acc[1]+1),            # SeqOp: update sum & count per partition
    lambda acc1, acc2: (acc1[0]+acc2[0], acc1[1]+acc2[1])  # CombOp: merge partitions
)

print(f"Sum: {result[0]}, Count: {result[1]}")


**Expected Output**: `Sum: 55, Count: 10`

### Exercise 17: CoGroup Two RDDs
Create RDDs: `[('a', 1), ('b', 2), ('a', 3)]` and `[('a', 'x'), ('b', 'y'), ('c', 'z')]`, use `cogroup`.

In [ ]:
from pyspark import SparkContext

sc = SparkContext.getOrCreate()

rdd1 = sc.parallelize([('a',1), ('b',2), ('a',3)])   # First RDD
rdd2 = sc.parallelize([('a','x'), ('b','y'), ('c','z')])  # Second RDD

cogrouped = rdd1.cogroup(rdd2)                       # Group values by key from both RDDs
result = cogrouped.mapValues(lambda x: (list(x[0]), list(x[1])))  # Convert iterables to lists

print(result.collect())


**Expected Output**: `[('a', ([1, 3], ['x'])), ('b', ([2], ['y'])), ('c', ([], ['z']))]` (order may vary)

### Exercise 18: Sample RDD
Load `covid-19.csv`, take a 1% sample with replacement, and count records.

In [ ]:
from pyspark import SparkContext

sc = SparkContext.getOrCreate()

rdd = sc.textFile("covid-dataset/covid-data.csv")            # Load CSV
header = rdd.first()                             # Get header
data = rdd.filter(lambda row: row != header)    # Skip header

sampled = data.sample(True, 0.01)               # Take 1% sample with replacement
count = sampled.count()                          # Count sampled rows

print(f"Sampled records count: {count}")


**Expected Output**: `Sampled records count: 4381`

### Exercise 19: Cartesian Product
Create RDDs: `[1, 2, 3]` and `['a', 'b']`, compute Cartesian product.

In [ ]:
from pyspark import SparkContext

sc = SparkContext.getOrCreate()

rdd1 = sc.parallelize([1, 2, 3])      # First RDD
rdd2 = sc.parallelize(['a', 'b'])    # Second RDD

cartesian_rdd = rdd1.cartesian(rdd2)  # Compute Cartesian product of two RDDs

print(cartesian_rdd.collect())


**Expected Output**: `[(1, 'a'), (1, 'b'), (2, 'a'), (2, 'b'), (3, 'a'), (3, 'b')]`

### Exercise 20: Key-Value Pair Operations with FlatMapValues
Create an RDD from `[('a', '1 2 3'), ('b', '4 5')]`, split values with `flatMapValues`.

In [ ]:
from pyspark import SparkContext

sc = SparkContext.getOrCreate()

data = [('a','1 2 3'), ('b','4 5')]
rdd = sc.parallelize(data)                             # Create RDD of key-value pairs

split_rdd = rdd.flatMapValues(lambda v: v.split())     # Split values into multiple key-value pairs

print(split_rdd.collect())


**Expected Output**: `[('a', '1'), ('a', '2'), ('a', '3'), ('b', '4'), ('b', '5')]`

### Exercise 21: Compute Variance
Create an RDD from `[1, 2, 3, 4, 5]`, compute variance.

In [ ]:
from pyspark import SparkContext

sc = SparkContext.getOrCreate()

numbers = [1,2,3,4,5]
rdd = sc.parallelize(numbers)                     # Create RDD from list

total = rdd.sum()                                 # Sum of elements
count = rdd.count()                               # Number of elements
mean = total / count                              # Compute mean

sq_diff = rdd.map(lambda x: (x - mean)**2)       # Squared differences from mean
variance = sq_diff.sum() / (count - 1)           # Compute sample variance

print(f"Variance: {variance}")


**Expected Output**: `Variance: 2.5`

### Exercise 22: Intersection of Two RDDs
Create RDDs: `[1, 2, 3, 4]` and `[3, 4, 5, 6]`, find intersection.

In [ ]:
from pyspark import SparkContext

sc = SparkContext.getOrCreate()

rdd1 = sc.parallelize([1, 2, 3, 4])    # First RDD
rdd2 = sc.parallelize([3, 4, 5, 6])    # Second RDD

intersection_rdd = rdd1.intersection(rdd2)  # Get common elements

print(intersection_rdd.collect())


**Expected Output**: `[3, 4]` (order may vary)

### Exercise 23: Zip Two RDDs
Create RDDs: `['a', 'b', 'c']` and `[1, 2, 3]`, zip them.

In [ ]:
from pyspark import SparkContext

sc = SparkContext.getOrCreate()

rdd1 = sc.parallelize(['a','b','c'])  # First RDD
rdd2 = sc.parallelize([1,2,3])        # Second RDD (same number of elements)

zipped_rdd = rdd1.zip(rdd2)           # Pair elements by index from both RDDs

print(zipped_rdd.collect())


**Expected Output**: `[('a', 1), ('b', 2), ('c', 3)]`

#### Notes
- Save this notebook as `exercises/02_solution.ipynb`.